In [13]:
import torch
import model.transformer
import model.data as data
import model.constants as constants
import time
print("START: ", time.perf_counter())
if constants.DEBUG: print("Start of file", time.perf_counter())
transformer = model.transformer.Transformer()
if constants.DEBUG: print("Transform made", time.perf_counter())
dev_cutoff = int(0.9 * len(data.full_text)) # TODO add test split
train_data = data.full_text[:dev_cutoff]
dev_data = data.full_text[dev_cutoff:]
X_batch, Y_batch = data.GetRandomBatch(train_data)

START:  65227.618508708


In [14]:

for i in range(3):
    if constants.DEBUG: print("Start", time.perf_counter())
    X_batch, Y_batch = data.GetRandomBatch(train_data);
    if constants.DEBUG: print("Got batches", time.perf_counter())
    out = transformer.forward(X_batch)
    if constants.DEBUG: print("Forwad complete", time.perf_counter())
    B, T, C = out.shape
    loss = transformer.backward(out.view(B*T, C), Y_batch.view(B*T))
    if constants.DEBUG: print("Backward complete", time.perf_counter())
    if (i % 1 == 0):
        print(f"loss: {loss}")

loss: 4.687386989593506
loss: 4.151747703552246
loss: 3.756960868835449


In [15]:

for i in range(EPOCHS):
    X_batch, Y_batch = GetRandomBatch(train_data);
    out = transformer.forward(X_batch)
    B, T, C = out.shape
    loss = transformer.backward(out.view(B*T, C), Y_batch.view(B*T))
    if (i % 1 == 0):
        print(f"loss: {loss}", time.perf_counter())

NameError: name 'EPOCHS' is not defined

In [ ]:
def DecodeTokenList(token_list):
    return data.decode(token_list)


def TestModel(transformer, data_source, n_tokens=10):
    X_test, _ = data.GetRandomBatch(data_source)
    # X_test = X_batch
    context = X_test[0].clone()
    start_context = context.clone()

    print("START:")
    print(DecodeTokenList(context))

    generated_tokens = []

    for i in range(n_tokens):
        # Add fake batch dimension: [T] -> [1, T]
        context_batch = context.view(1, constants.CONTEXT_WINDOW_SIZE)

        # Your transformer currently expects N_BATCHES exactly,
        # so repeat the same context N_BATCHES times.
        context_batch = context_batch.repeat(constants.N_BATCHES, 1)

        # Predict next token from the final position in the first batch row
        with torch.no_grad():
            logits = transformer.forward(context_batch)
        pred = logits[0, -1].argmax(dim=-1)

        generated_tokens.append(pred)

        # Slide context window left and append prediction
        context = torch.cat([context[1:], pred.view(1)], dim=0)

        print(f"PRED {i + 1}: {data.untokenizer_map[int(pred)]!r}")

    print("\nGENERATED:")
    print(DecodeTokenList(generated_tokens))

    print("\nFULL:")
    print(DecodeTokenList(start_context) + DecodeTokenList(generated_tokens))

TestModel(transformer, train_data, n_tokens=10)

START:
re measure to your woes,
I come 
PRED 1: 'o'


KeyboardInterrupt: 